# Code review: ревью чужого кода и работа по замечаниям

**Учебное задание (2 ч).** Среда — Google Colab, язык — Java (каждый пример — один файл, запуск `java Файл.java`). Работа парная: вторая часть выполняется на репозитории однокурсника.

Рефакторинг кода : как прочитать чужой код, сформулировать замечания, которые автор примет, и как принять замечания к своему коду. Это отдельный навык, и он проверяется на каждом вашем pull request в течение семестра.

**Что получится на выходе**

| Артефакт | Где |
|---|---|
| Таблица замечаний к выданному фрагменту | `ReviewNotes.csv` |
| Исправленный по замечаниям код с тестами | `ShiftPlanFixed.java` |
| Ревью pull request однокурсника, не менее пяти замечаний | GitHub, ссылка в отчёте |
| Правки своего кода по полученному ревью | GitHub, ссылка в отчёте |

Исследования и практика разработки сходятся в одном: ревью находит дефекты дешевле, чем тестирование, и втрое дешевле, чем исправление после выпуска. Но плохо сформулированное замечание не исправляет код, а портит отношения в команде, после чего ревью превращается в штамповку «одобряю». Поэтому формулировка здесь — предмет изучения наравне с содержанием.

---
## 1. Подготовка

In [14]:
!java -version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [15]:
import csv, os, re, subprocess, datetime
import pandas as pd

os.makedirs("ревью", exist_ok=True)
print("Среда готова.")

Среда готова.


---
## 2. Чек-лист ревью

Ревью без чек-листа превращается в вычитку стиля: рецензент замечает отступы и пропускает ошибку в условии. Порядок просмотра — сверху вниз: сначала то, что ломает работу системы, в конце — вкусовое.

| Категория | На что смотреть | Серьёзность по умолчанию |
|---|---|---|
| Корректность | ошибки в условиях и границах, сравнение объектов, деление на ноль, неинициализированные значения | blocker |
| Поведение при ошибках | проглоченные исключения, отсутствие проверки входных данных, незакрытые ресурсы | blocker / major |
| Данные и безопасность | пароли и строки подключения в коде, конкатенация SQL, вывод персональных данных в журнал | blocker |
| Тесты | изменённое поведение не покрыто тестом, тест проверяет реализацию вместо поведения | major |
| Проектирование | нарушение единственной ответственности, ветвление по типу вместо полиморфизма, дублирование | major |
| Читаемость | неинформативные имена, магические числа, комментарий противоречит коду | minor |
| Оформление | форматирование, порядок импортов, лишние пустые строки | nit |

**Шкала серьёзности.** `blocker` — слияние невозможно; `major` — нужно исправить в этом же запросе; `minor` — исправить желательно; `nit` — вкусовое, автор вправе не согласиться. Замечание без указанной серьёзности заставляет автора гадать, что обязательно, а что нет.

### Как формулируется замечание

Замечание состоит из трёх частей: **что** (конкретное место), **почему** (последствие или принцип), **предложение** (как можно сделать). Говорят о коде, а не об авторе.

| Плохо | Почему плохо | Хорошо |
|---|---|---|
| «Это ужасно написано» | оценка автора, нет предмета | «Строка 24: сравнение строк через `==` сравнивает ссылки, а не значения — при другом источнике данных условие перестанет срабатывать. Предлагаю `equals`» |
| «Переименуй переменные» | нет конкретики и причины | «Строки 18–30: имена `t`, `h`, `r` не говорят о величинах; предлагаю `вагонТип`, `часыПростоя`, `ставка` — расчёт станет читаться без комментариев» |
| «А если отрицательное?» | вопрос вместо замечания | «Строка 19: при отрицательном числе часов метод вернёт отрицательную плату. Предлагаю отбрасывать такие значения исключением `IllegalArgumentException`» |
| «Перепиши на паттерн» | требование без обоснования | «Строки 20–34: при добавлении нового типа вагона придётся править этот метод (нарушение принципа открытости/закрытости). Предлагаю стратегию расчёта, как в ЛР по паттернам — но это `major`, можно отдельным запросом» |

---
## 3. Образец: разбор выданного фрагмента

Ниже — метод расчёта платы за простой вагонов. Код работает и даёт правдоподобные числа, поэтому глазами «запускается — значит нормально» дефекты не видны. Запустите его и прочитайте внимательно, **прежде** чем смотреть эталонный разбор.

In [16]:
%%writefile DemurrageLegacy.java
public class DemurrageLegacy {

    public static void main(String[] args) {
        calc("крытый", 30, 2.5);
        calc("цистерна", 30, 2.5);
        calc("платформа", -5, 2.5);
    }

    // Расчет платы за простой вагона сверх нормы. Норма - одни сутки.
    public static double calc(String t, double h, double r) {
        double s = 0;
        if (t == "крытый") {
            s = (h - 24) * r * 1.0;
        } else if (t == "цистерна") {
            s = (h - 24) * r * 1.15;
        } else {
            s = (h - 24) * r * 1.0;
        }
        if (s > 1000) {
            s = s * 0.9;   // скидка
        }
        System.out.println("Плата: " + s);
        return s;
    }
}

Overwriting DemurrageLegacy.java


In [17]:
!java DemurrageLegacy.java

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Плата: 15.0
Плата: 17.25
Плата: -72.5


### 3.1 Эталонная таблица замечаний

Обратите внимание на три вещи: замечаний немного, каждое указывает место, каждое содержит предложение. Девять замечаний на тридцать строк — нормальная плотность; тридцать замечаний автор просто не станет читать.

In [18]:
эталонные_замечания = [
    dict(Строка="22", Категория="Корректность", Серьёзность="blocker",
         Замечание="Сравнение строк через == сравнивает ссылки. Сейчас условие срабатывает случайно, из-за пула строковых литералов; при получении типа вагона из файла или базы данных расчёт молча уйдёт в ветку «иначе»",
         Предложение="Использовать equals, а лучше перечисление ТипВагона вместо строки"),
    dict(Строка="20", Категория="Поведение при ошибках", Серьёзность="blocker",
         Замечание="При числе часов меньше нормы плата получается отрицательной: вызов calc(\"платформа\", -5, 2.5) возвращает отрицательное значение и печатает его как плату",
         Предложение="Проверять аргументы: отрицательные часы и ставку отклонять IllegalArgumentException, простой меньше нормы считать нулевой платой"),
    dict(Строка="22-28", Категория="Проектирование", Серьёзность="major",
         Замечание="Ветвление по типу вагона: добавление нового типа требует правки этого метода (нарушение принципа открытости/закрытости)",
         Предложение="Вынести коэффициент в перечисление или в стратегию расчёта, как в работе по паттернам"),
    dict(Строка="22-28", Категория="Проектирование", Серьёзность="major",
         Замечание="Формула (h - 24) * r * коэффициент повторена трижды: при изменении методики править придётся в трёх местах",
         Предложение="Вычислить базу один раз, а по типу вагона выбирать только коэффициент"),
    dict(Строка="31", Категория="Читаемость", Серьёзность="major",
         Замечание="Числа 24, 1.15, 1000 и 0.9 ничего не говорят читателю и не прослеживаются до документа, которым установлены",
         Предложение="Вынести в именованные константы: НОРМА_ПРОСТОЯ_ЧАСОВ, КОЭФФИЦИЕНТ_ЦИСТЕРНЫ, ПОРОГ_СКИДКИ, КОЭФФИЦИЕНТ_СКИДКИ"),
    dict(Строка="18", Категория="Проектирование", Серьёзность="major",
         Замечание="Метод одновременно считает плату и печатает её: расчёт нельзя переиспользовать ни в отчёте, ни в тесте без побочного вывода",
         Предложение="Метод возвращает значение, печать выполняет вызывающий код"),
    dict(Строка="18", Категория="Читаемость", Серьёзность="minor",
         Замечание="Имена t, h, r, s не раскрывают величины, из-за чего формулу приходится расшифровывать",
         Предложение="типВагона, часыПростоя, ставкаЗаЧас, плата — расчёт станет самодокументируемым"),
    dict(Строка="19", Категория="Корректность", Серьёзность="minor",
         Замечание="Денежная величина хранится в double: при суммировании по ведомости накопится ошибка округления",
         Предложение="BigDecimal либо хранение в копейках целым числом; решение зафиксировать в комментарии"),
    dict(Строка="17", Категория="Тесты", Серьёзность="major",
         Замечание="Для метода нет ни одного теста, хотя он содержит четыре ветви и пороговое условие",
         Предложение="Добавить тесты на каждую ветвь и на границу 24 часа и порог скидки"),
]

эталон = pd.DataFrame(эталонные_замечания)
print("Замечаний в эталоне:", len(эталон))
print("По серьёзности:", эталон["Серьёзность"].value_counts().to_dict())
эталон[["Строка", "Категория", "Серьёзность", "Замечание"]]

Замечаний в эталоне: 9
По серьёзности: {'major': 5, 'blocker': 2, 'minor': 2}


,Строка,Категория,Серьёзность,Замечание
0,22,Корректность,blocker,Сравнение строк через == сравнивает ссылки. Се...
1,20,Поведение при ошибках,blocker,При числе часов меньше нормы плата получается ...
2,22-28,Проектирование,major,Ветвление по типу вагона: добавление нового ти...
3,22-28,Проектирование,major,Формула (h - 24) * r * коэффициент повторена т...
4,31,Читаемость,major,"Числа 24, 1.15, 1000 и 0.9 ничего не говорят ч..."
5,18,Проектирование,major,Метод одновременно считает плату и печатает её...
6,18,Читаемость,minor,"Имена t, h, r, s не раскрывают величины, из-за..."
7,19,Корректность,minor,Денежная величина хранится в double: при сумми...
8,17,Тесты,major,"Для метода нет ни одного теста, хотя он содерж..."


### 3.2 Чего в эталоне нет

В разборе намеренно отсутствуют замечания двух видов, и это тоже часть образца:

* **вкусовые требования без обоснования** — «перепиши циклом for», «не люблю else» — такие замечания тратят время и обесценивают остальные;
* **переписывание кода за автора** — рецензент предлагает направление, решение остаётся за автором. Если рецензент сам пишет исправление, ответственность за код размывается.

**Вопрос.** В эталоне нет замечания о том, что `calc` объявлен `static`. Должно ли оно быть? Обоснуйте.

**Ответ:** Замечание о модификаторе `static` в данном случае **не нужно** (либо максимум уровня `nit`). Метод `calc` является чистой функцией расчёта: он не использует состояние объекта и зависит только от переданных параметров, поэтому объявление его как `static` вполне оправдано. Требовать убрать `static` имело бы смысл только при переходе к объектной модели или паттерну Strategy, а без такого контекста это было бы вкусовым придирательством.

---
## 4. Задача 1: самостоятельный разбор

Второй фрагмент — формирование наряда локомотивных бригад на смену. Дефекты здесь другого рода: не формула, а работа с данными, ресурсами и исключениями. Запустите, изучите и составьте таблицу замечаний.

In [19]:
%%writefile ShiftPlanLegacy.java
import java.io.FileWriter;
import java.text.SimpleDateFormat;
import java.util.ArrayList;
import java.util.Date;
import java.util.List;

public class ShiftPlanLegacy {

    static SimpleDateFormat dateFormat = new SimpleDateFormat("dd.MM.yyyy");
    static List<String> log = new ArrayList<>();

    public static void main(String[] args) {
        List<Brigade> brigades = new ArrayList<>();
        brigades.add(new Brigade("Иванов", 11.0, true));
        brigades.add(new Brigade("Петров", 0.0, true));
        brigades.add(new Brigade("Сидоров", 4.5, false));
        brigades.add(new Brigade("Козлов", 8.0, true));

        System.out.println(make(brigades, 3));
        System.out.println("Журнал: " + log);
    }

    public static String make(List<Brigade> brigades, int need) {
        for (Brigade brigade : brigades) {
            if (brigade.hoursPerShift > 10.0 || brigade.admitted == false) {
                brigades.remove(brigade);
            }
        }

        String result = "";
        for (int i = 0; i < need; i++) {
            result = result + brigades.get(i).lastName + "; ";
        }

        try {
            FileWriter writer = new FileWriter("наряд.txt");
            writer.write(result);
        } catch (Exception e) {
        }

        log.add(dateFormat.format(new Date()) + " наряд сформирован");
        return result;
    }
}

class Brigade {
    String lastName;
    double hoursPerShift;
    boolean admitted;

    Brigade(String lastName, double hoursPerShift, boolean admitted) {
        this.lastName = lastName;
        this.hoursPerShift = hoursPerShift;
        this.admitted = admitted;
    }
}

Overwriting ShiftPlanLegacy.java


In [20]:
!java ShiftPlanLegacy.java

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Exception in thread "main" java.util.ConcurrentModificationException
	at java.base/java.util.ArrayList$Itr.checkForComodification(ArrayList.java:1095)
	at java.base/java.util.ArrayList$Itr.next(ArrayList.java:1049)
	at ShiftPlanLegacy.make(ShiftPlanLegacy.java:24)
	at ShiftPlanLegacy.main(ShiftPlanLegacy.java:19)


**Подсказка, куда смотреть.** Запуск может завершиться исключением, а может и не завершиться — это зависит от данных, и само по себе является дефектом. Проверьте: изменение списка во время обхода; что произойдёт при `нужно` больше числа оставшихся бригад; закрывается ли файл; что происходит с исключением; переживёт ли статическое состояние одновременный вызов из двух потоков; сравнение вещественных чисел.

### 4.1 Ваша таблица замечаний

Заполните список ниже: **не менее восьми** замечаний, из них **не менее трёх** с серьёзностью `blocker` или `major`. В каждом замечании обязательны место, последствие и предложение. Поле «Предложение» короче двадцати символов самопроверка не примет.

In [21]:
# 4.1 Ваши замечания. Формат тот же, что в эталоне.
мои_замечания = [
    dict(Строка="24-28", Категория="Корректность", Серьёзность="blocker",
         Замечание="Удаление элементов из списка brigades внутри цикла for-each вызывает ConcurrentModificationException и к тому же портит исходный список вызывающего кода",
         Предложение="Не изменять переданный список, а отбирать подходящие бригады в новый список (через цикл или Stream API)"),
    dict(Строка="31-33", Категория="Поведение при ошибках", Серьёзность="blocker",
         Замечание="Отсутствует проверка входных данных: если need больше числа подходящих бригад, need <= 0 или brigades == null, возникнет IndexOutOfBoundsException или NullPointerException",
         Предложение="Проверять аргументы (brigades != null, need > 0) и при нехватке доступных бригад выбрасывать IllegalArgumentException"),
    dict(Строка="36-37", Категория="Поведение при ошибках", Серьёзность="blocker",
         Замечание="Поток FileWriter открывается, но никогда не закрывается, что ведёт к утечке ресурсов и возможной потере данных в буфере",
         Предложение="Использовать конструкцию try-with-resources (try (FileWriter writer = ...)), чтобы файл гарантированно закрывался"),
    dict(Строка="38-39", Категория="Поведение при ошибках", Серьёзность="blocker",
         Замечание="Пустой блок catch (Exception e) {} молча проглатывает любые ошибки записи в файл — программа делает вид, что всё прошло успешно",
         Предложение="Обрабатывать ошибку ввода-вывода: фиксировать причину и пробрасывать осмысленное исключение (например, IOException или UncheckedIOException)"),
    dict(Строка="9-10", Категория="Корректность", Серьёзность="major",
         Замечание="Статические поля SimpleDateFormat (не потокобезопасен) и ArrayList log при одновременном вызове из нескольких потоков приведут к гонке данных",
         Предложение="Заменить SimpleDateFormat на потокобезопасный DateTimeFormatter (java.time), а журнал сделать полем экземпляра объекта"),
    dict(Строка="23-42", Категория="Проектирование", Серьёзность="major",
         Замечание="Метод make нарушает принцип единственной ответственности: одновременно фильтрует бригады, формирует строку, пишет в жёстко заданный файл и ведёт журнал",
         Предложение="Разделить логику: вынести формирование наряда в отдельный чистый метод, а запись в файл и путь к файлу отделить"),
    dict(Строка="25", Категория="Читаемость", Серьёзность="minor",
         Замечание="Магическое число 10.0 не объясняет норму часов, а запись brigade.admitted == false избыточна",
         Предложение="Вынести 10.0 в именованную константу MAX_SHIFT_HOURS, а проверку допуска записать лаконично как !brigade.admitted"),
    dict(Строка="30-33", Категория="Читаемость", Серьёзность="minor",
         Замечание="Конкатенация строк через '+' в цикле создаёт лишние промежуточные объекты String, а имя метода make не раскрывает его назначение",
         Предложение="Использовать StringBuilder или String.join для сборки результата и переименовать метод в buildShiftPlan"),
    dict(Строка="23", Категория="Тесты", Серьёзность="major",
         Замечание="Для метода нет ни одного теста, хотя в нём есть фильтрация, граничное условие 10.0 часов и обработка ошибок",
         Предложение="Добавить самопроверки на корректный отбор, граничное значение 10.0 часов, нехватку бригад и некорректные аргументы"),
]

ревью = pd.DataFrame(мои_замечания)
ревью.to_csv("ReviewNotes.csv", index=False, encoding="utf-8")
print("Записано замечаний:", len(ревью))
ревью

Записано замечаний: 9


,Строка,Категория,Серьёзность,Замечание,Предложение
0,24-28,Корректность,blocker,Удаление элементов из списка brigades внутри ц...,"Не изменять переданный список, а отбирать подх..."
1,31-33,Поведение при ошибках,blocker,Отсутствует проверка входных данных: если need...,"Проверять аргументы (brigades != null, need > ..."
2,36-37,Поведение при ошибках,blocker,"Поток FileWriter открывается, но никогда не за...",Использовать конструкцию try-with-resources (t...
3,38-39,Поведение при ошибках,blocker,Пустой блок catch (Exception e) {} молча прогл...,Обрабатывать ошибку ввода-вывода: фиксировать ...
4,9-10,Корректность,major,Статические поля SimpleDateFormat (не потокобе...,Заменить SimpleDateFormat на потокобезопасный ...
5,23-42,Проектирование,major,Метод make нарушает принцип единственной ответ...,Разделить логику: вынести формирование наряда ...
6,25,Читаемость,minor,Магическое число 10.0 не объясняет норму часов...,Вынести 10.0 в именованную константу MAX_SHIFT...
7,30-33,Читаемость,minor,Конкатенация строк через '+' в цикле создаёт л...,Использовать StringBuilder или String.join для...
8,23,Тесты,major,"Для метода нет ни одного теста, хотя в нём ест...","Добавить самопроверки на корректный отбор, гра..."


---
## 5. Задача 2: исправление по замечаниям

Исправьте `ShiftPlanLegacy` по **своим же** замечаниям в файле `ShiftPlanFixed.java`. Требования:

1. Внешнее поведение при корректных данных сохраняется: тот же состав наряда при тех же исходных данных. Это фиксируется характеризующим тестом — сначала запишите ожидаемый результат, полученный от исходной версии, затем правьте код.
2. Устранены все замечания серьёзности `blocker` и `major` из вашей таблицы.
3. Некорректные входные данные приводят к осмысленному исключению, а не к `IndexOutOfBoundsException` или молчаливому пропуску.
4. Файл содержит не менее пяти самопроверок методом `check`, как в предыдущих работах: `[OK]` / `[FAIL]` и `AssertionError` при неуспехе.
5. Первым в файле идёт открытый класс `ShiftPlanFixed` с методом `main`.

Рефакторинг ради красоты здесь не требуется: каждое изменение должно быть прослеживаемо до строки вашей таблицы замечаний.

In [22]:
%%writefile ShiftPlanFixed.java
// 5.1 Исправленный вариант ShiftPlanLegacy (с учётом замечаний ревьюера)
import java.io.FileWriter;
import java.io.IOException;
import java.io.UncheckedIOException;
import java.nio.file.Files;
import java.nio.file.Path;
import java.time.LocalDate;
import java.time.format.DateTimeFormatter;
import java.util.ArrayList;
import java.util.Collections;
import java.util.List;

public class ShiftPlanFixed {

    private static final DateTimeFormatter DATE_FORMATTER = DateTimeFormatter.ofPattern("dd.MM.yyyy");
    private static final double MAX_SHIFT_HOURS = 10.0;

    // Потокобезопасный журнал экземпляра (по замечанию minor)
    private final List<String> log = Collections.synchronizedList(new ArrayList<>());

    private static int failedChecks = 0;

    public static void main(String[] args) {
        ShiftPlanFixed planner = new ShiftPlanFixed();

        List<Brigade> brigades = new ArrayList<>();
        brigades.add(new Brigade("Иванов", 11.0, true));
        brigades.add(new Brigade("Петров", 0.0, true));
        brigades.add(new Brigade("Сидоров", 4.5, false));
        brigades.add(new Brigade("Козлов", 8.0, true));

        System.out.println("--- Самопроверки исправленного кода ---");

        // Проверка 1 (характеризующий тест): корректный состав наряда и сохранность исходного списка
        String plan = planner.buildShiftPlan(brigades, 2);
        check("корректный состав наряда (Петров; Козлов; )", "Петров; Козлов; ".equals(plan));
        check("исходный список бригад не изменён (осталось 4 элемента)", brigades.size() == 4);

        // Проверка 2: граничное значение 10.0 часов (бригада должна допускаться)
        List<Brigade> boundaryList = List.of(new Brigade("Смирнов", 10.0, true));
        check("граничное значение 10.0 часов допускается в наряд",
                "Смирнов; ".equals(planner.buildShiftPlan(boundaryList, 1)));

        // Проверка 3: нехватка бригад вызывает IllegalArgumentException
        boolean shortageCaught = false;
        try {
            planner.buildShiftPlan(brigades, 3);
        } catch (IllegalArgumentException e) {
            shortageCaught = true;
        }
        check("при нехватке доступных бригад выбрасывается IllegalArgumentException", shortageCaught);

        // Проверка 4: некорректные аргументы (need <= 0 или null) вызывают IllegalArgumentException
        boolean invalidArgsCaught = false;
        try {
            planner.buildShiftPlan(brigades, 0);
        } catch (IllegalArgumentException e) {
            invalidArgsCaught = true;
        }
        boolean nullListCaught = false;
        try {
            planner.buildShiftPlan(null, 1);
        } catch (IllegalArgumentException e) {
            nullListCaught = true;
        }
        check("при некорректных аргументах (0 или null) выбрасывается IllegalArgumentException",
                invalidArgsCaught && nullListCaught);

        // Проверка 5: валидация в конструкторе Brigade (по замечанию major)
        boolean invalidBrigadeCaught = false;
        try {
            new Brigade("   ", -1.0, true);
        } catch (IllegalArgumentException e) {
            invalidBrigadeCaught = true;
        }
        check("конструктор Brigade отклоняет пустую фамилию и отрицательные часы", invalidBrigadeCaught);

        // Проверка 6: запись в файл и фиксация события в журнале (по замечанию major)
        Path tempFile = Path.of("наряд.txt");
        planner.saveShiftPlan(plan, tempFile);
        boolean fileWrittenCorrectly = false;
        try {
            fileWrittenCorrectly = plan.equals(Files.readString(tempFile));
        } catch (IOException e) {
            fileWrittenCorrectly = false;
        }
        check("наряд корректно сохранён в файл", fileWrittenCorrectly);
        check("журнал экземпляра содержит запись после сохранения наряда", planner.getLog().size() == 1);

        if (failedChecks > 0) {
            throw new AssertionError("не пройдено проверок: " + failedChecks);
        }
        System.out.println();
        System.out.println("Все проверки пройдены.");
    }

    /**
     * Чистый метод формирования наряда (без побочных эффектов записи в файл и журнал).
     */
    public String buildShiftPlan(List<Brigade> brigades, int requiredCount) {
        if (brigades == null) {
            throw new IllegalArgumentException("Список бригад не может быть null");
        }
        if (requiredCount <= 0) {
            throw new IllegalArgumentException("Требуемое количество бригад должно быть больше 0");
        }

        List<Brigade> available = new ArrayList<>();
        for (Brigade brigade : brigades) {
            if (brigade != null && brigade.getHoursPerShift() <= MAX_SHIFT_HOURS && brigade.isAdmitted()) {
                available.add(brigade);
            }
        }

        if (available.size() < requiredCount) {
            throw new IllegalArgumentException(
                    "Недостаточно доступных бригад: требуется " + requiredCount + ", доступно " + available.size());
        }

        StringBuilder result = new StringBuilder();
        for (int i = 0; i < requiredCount; i++) {
            result.append(available.get(i).getLastName()).append("; ");
        }
        return result.toString();
    }

    /**
     * Запись наряда в файл с try-with-resources и фиксация события в журнале.
     */
    public void saveShiftPlan(String shiftPlan, Path outputPath) {
        try (FileWriter writer = new FileWriter(outputPath.toFile())) {
            writer.write(shiftPlan);
        } catch (IOException e) {
            throw new UncheckedIOException("Ошибка записи наряда в файл: " + outputPath, e);
        }
        log.add(LocalDate.now().format(DATE_FORMATTER) + " наряд сформирован");
    }

    public List<String> getLog() {
        return List.copyOf(log);
    }

    static void check(String checkName, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + checkName);
        if (!passed) {
            failedChecks++;
        }
    }
}

class Brigade {
    private final String lastName;
    private final double hoursPerShift;
    private final boolean admitted;

    Brigade(String lastName, double hoursPerShift, boolean admitted) {
        if (lastName == null || lastName.isBlank()) {
            throw new IllegalArgumentException("Фамилия бригады не может быть пустой");
        }
        if (hoursPerShift < 0) {
            throw new IllegalArgumentException("Часы работы за смену не могут быть отрицательными");
        }
        this.lastName = lastName;
        this.hoursPerShift = hoursPerShift;
        this.admitted = admitted;
    }

    public String getLastName() {
        return lastName;
    }

    public double getHoursPerShift() {
        return hoursPerShift;
    }

    public boolean isAdmitted() {
        return admitted;
    }
}

Overwriting ShiftPlanFixed.java


In [23]:
!java ShiftPlanFixed.java

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
--- Самопроверки исправленного кода ---
[OK]   корректный состав наряда (Петров; Козлов; )
[OK]   исходный список бригад не изменён (осталось 4 элемента)
[OK]   граничное значение 10.0 часов допускается в наряд
[OK]   при нехватке доступных бригад выбрасывается IllegalArgumentException
[OK]   при некорректных аргументах (0 или null) выбрасывается IllegalArgumentException
[OK]   конструктор Brigade отклоняет пустую фамилию и отрицательные часы
[OK]   наряд корректно сохранён в файл
[OK]   журнал экземпляра содержит запись после сохранения наряда

Все проверки пройдены.


**Таблица прослеживаемости.** Заполните соответствие «замечание → что изменено». Именно она, а не объём правок, показывает качество работы.

| № замечания | Серьёзность | Что изменено в коде |
|---|---|---|
| 1 | blocker | Удаление из списка внутри `for-each` заменено на отбор подходящих бригад в новый список `available` — исходный список не портится |
| 2 | blocker | Добавлены проверки `brigades == null`, `requiredCount <= 0` и нехватки доступных бригад с выбросом `IllegalArgumentException` |
| 3 | blocker | Открытие `FileWriter` переведено на конструкцию `try-with-resources`, чтобы файл гарантированно закрывался |
| 4 | blocker | Пустой блок `catch` заменён на перехват `IOException` с пробросом `UncheckedIOException` и сохранением причины |
| 5 | major | `SimpleDateFormat` заменён на потокобезопасный `DateTimeFormatter`, а журнал `log` сделан полем экземпляра вместо `static` |
| 6 | major | Логика разделена на чистый метод формирования наряда `buildShiftPlan` и отдельный метод записи в файл `saveShiftPlan` с передачей `Path` |
| 7 | minor | Магическое число `10.0` вынесено в константу `MAX_SHIFT_HOURS`, проверка допуска упрощена до `brigade.admitted` |
| 8 | minor | Конкатенация строк в цикле заменена на `StringBuilder`, метод `make` переименован в `buildShiftPlan` |
| 9 | major | В `main` добавлены самопроверки: характеризующий тест, граничное значение `10.0` ч, нехватка бригад, некорректные аргументы, журнал и запись в файл |

---
## 6. Задача 3: ревью на GitHub

Эта часть выполняется в паре на действующих репозиториях курса.

**Как рецензент**

1. Откройте pull request однокурсника (ваша пара назначается преподавателем).
2. Начните ревью: вкладка *Files changed* → *Review changes* → *Start a review*. Замечания оставляйте **привязанными к строкам**, а не одним комментарием ко всему запросу.
3. Оставьте не менее пяти замечаний, из них хотя бы одно `blocker` или `major`. Серьёзность указывайте префиксом в тексте: `blocker:`, `major:`, `minor:`, `nit:`.
4. Завершите ревью с решением *Request changes* или *Comment*. Решение *Approve* при наличии незакрытых `blocker` недопустимо.
5. Отметьте в ревью хотя бы одно удачное решение автора. Ревью, состоящее только из претензий, перестают читать.

**Как автор**

6. На каждое полученное замечание ответьте: принято и исправлено, либо обоснованный отказ. Молчание ответом не считается.
7. Внесите правки отдельными коммитами со ссылкой на замечание.
8. Закройте обсуждения кнопкой *Resolve conversation* — только те, что действительно исправлены.

**Что фиксируется в отчёте:** ссылка на оставленное вами ревью, ссылка на ваш запрос с полученным ревью и внесёнными правками.

In [24]:
# 6.1 Ссылки на выполненное ревью (понадобятся самопроверке)
ССЫЛКА_НА_МОЁ_РЕВЬЮ = "[https://github.com/VeronikaZmushko/Lab8_Code_Review/pull/1]"      #@param {type:"string"}
ССЫЛКА_НА_МОЙ_PR = "https://github.com/Violaveta/docker_lab/pull/4"         #@param {type:"string"}
ЗАМЕЧАНИЙ_ОСТАВЛЕНО = 5       #@param {type:"integer"}
ЗАМЕЧАНИЙ_ПОЛУЧЕНО = 5        #@param {type:"integer"}
ЗАМЕЧАНИЙ_ПРИНЯТО = 5         #@param {type:"integer"}

print("Ревью:", ССЫЛКА_НА_МОЁ_РЕВЬЮ or "не указано")
print("Свой запрос:", ССЫЛКА_НА_МОЙ_PR or "не указан")

Ревью: [https://github.com/VeronikaZmushko/Lab8_Code_Review/pull/1]
Свой запрос: https://github.com/Violaveta/docker_lab/pull/4


---
## 7. Разбор и выводы

### 7.1 Сравнение своей таблицы с эталоном подхода

| Вопрос | Ответ |
|---|---|
| Сколько замечаний вы нашли и сколько из них `blocker`? | Найдено 9 замечаний, из них 4 уровня `blocker` (удаление в цикле `for-each`, отсутствие проверки аргументов, незакрытый `FileWriter` и пустой блок `catch`). |
| Какое замечание вы нашли не сразу, и что помешало увидеть его раньше? | Непотокобезопасность статических полей `SimpleDateFormat` и `log` — при обычном однопоточном запуске код работает без ошибок, поэтому проблема видна только при внимательном чтении. |
| Было ли среди ваших замечаний вкусовое, выданное за существенное? | Нет, все замечания уровня `blocker` и `major` относятся к реальным сбоям, утечкам ресурсов, потокобезопасности и архитектуре, а стилистические правки отнесены к `minor`. |
| Какое из полученных вами замечаний вы отклонили и на каком основании? | Все конструктивные замечания приняты; вкусовые замечания уровня `nit` без влияния на корректность и читаемость можно обоснованно отклонять. |

### 7.2 Контрольные вопросы

1. Чем ревью отличается от тестирования и от статического анализа? Что из этого дешевле и почему ни одно не заменяет остальные.
   **Ответ:** Статический анализатор автоматически ищет шаблонные ошибки и нарушения стиля; тесты проверяют заявленное поведение на конкретных сценариях; ревью — это осмысленная проверка кода человеком на логику, архитектуру, читаемость и соответствие требованиям. Дешевле всего статический анализ и ревью на раннем этапе (дефекты находятся до попадания в основную ветку). Ни один метод не заменяет остальные, так как анализатор не понимает бизнес-логику, тесты не ловят плохую архитектуру и нечитаемые имена, а человек может пропустить краевой случай, который поймает автотест.
2. Какие виды дефектов статический анализатор найдёт лучше человека, а какие не найдёт никогда?
   **Ответ:** Анализатор лучше находит неиспользуемые переменные, незакрытые потоки, потенциальные `NullPointerException`, нарушения форматирования и типовые уязвимости. Он никогда не найдёт неверно понятое бизнес-требование, неудачную архитектуру, бессмысленные имена переменных в контексте предметной области или то, что комментарий расходится со смыслом задачи.
3. Из каких трёх частей состоит полезное замечание? Приведите своё замечание, переписанное из плохой формулировки в хорошую.
   **Ответ:** Полезное замечание состоит из трёх частей: **место** (файл и строка), **причина/последствие** (что сломается или какой принцип нарушен) и **предложение** (как исправить). Пример: вместо плохого «Убери пустой catch, так нельзя» → хорошее: «Строки 38–39: пустой блок `catch (Exception e) {}` молча проглатывает ошибки записи в файл, из-за чего программа скрывает сбой. Предлагаю перехватывать `IOException` и пробрасывать `UncheckedIOException` с описанием причины».
4. Почему `Approve` при незакрытом `blocker` опаснее, чем отсутствие ревью вообще?
   **Ответ:** Потому что `Approve` создаёт ложное чувство безопасности: команда и автор считают, что код проверен и надёжен, хотя в основную ветку попадает критический дефект. Кроме того, формальное одобрение с ошибками обесценивает саму культуру ревью в команде.
5. Что такое характеризующий тест и зачем он нужен перед исправлением чужого кода?
   **Ответ:** Характеризующий тест фиксирует текущее фактическое поведение кода на корректных данных до начала правок. Он нужен как страховка при рефакторинге: после всех исправлений тест гарантирует, что мы не сломали работающую логику.
6. Почему крупный pull request получает менее качественное ревью, чем несколько мелких?
   **Ответ:** При большом объёме изменений внимание рецензента быстро рассеивается, ему сложнее удерживать контекст в голове, и ревью скатывается либо к поверхностным придиркам к стилю, либо к формальному `Approve` без глубокого разбора. Небольшие PR проверяются быстрее, внимательнее и по существу.

### 7.3 Три вывода по работе

1. Ревью по чек-листу сверху вниз (от критических ошибок к оформлению) позволяет не тратить время на мелочи и находить реальные дефекты вроде утечки ресурсов, `ConcurrentModificationException` и проглоченных исключений.
2. Конструктивная формулировка замечаний с указанием места, последствия, предложения и уровня серьёзности делает проверку полезной для автора и помогает сохранить рабочую атмосферу в команде.
3. Связка «таблица замечаний → характеризующие тесты → исправления с прослеживаемостью» делает рефакторинг предсказуемым и доказуемым.

---
## 8. Чек-лист и самопроверка

- [ ] Образец изучен, отвечен вопрос п. 3.2
- [ ] Таблица замечаний заполнена: не менее восьми записей, все поля непустые
- [ ] Не менее трёх замечаний серьёзности `blocker` или `major`
- [ ] В каждом замечании указано место, последствие и предложение
- [ ] `ShiftPlanFixed.java` написан, самопроверки проходят
- [ ] Заполнена таблица прослеживаемости «замечание → изменение»
- [ ] На GitHub оставлено ревью, не менее пяти замечаний, привязанных к строкам
- [ ] На полученные замечания даны ответы, обсуждения закрыты по существу
- [ ] Отвечены контрольные вопросы и сформулированы выводы
- [ ] Отчёт самопроверки вставлен в описание pull request

In [25]:
#@title Запустить самопроверку и получить отчёт
import datetime, os, re, subprocess

import pandas as pd

ФИО_исполнителя = "Дёшева Виолетта Евгеньевна"  #@param {type:"string"}
Группа_исполнителя = "ГИ-41"  #@param {type:"string"}
ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА = True  #@param {type:"boolean"}
ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ = True  #@param {type:"boolean"}

ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ = ["Строка", "Категория", "Серьёзность", "Замечание", "Предложение"]
ВЕСОМЫЕ = {"blocker", "major"}
ОБРАЗЕЦ_ССЫЛКИ = re.compile(r"https://github\.com/[\w.-]+/[\w.-]+/pull/\d+")

ошибки = []
строки_отчёта = [
    "### Отчёт самопроверки «Code review»",
    "",
    "| Параметр | Значение |",
    "|---|---|",
    "| Исполнитель | %s, группа %s |" % (ФИО_исполнителя or "НЕ УКАЗАН", Группа_исполнителя or "НЕ УКАЗАНА"),
    "| Дата отчёта (UTC) | %s |" % datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M"),
]

# --- 1. таблица замечаний ---
if not os.path.exists("ReviewNotes.csv"):
    ошибки.append("нет файла ReviewNotes.csv")
    строки_отчёта.append("| Таблица замечаний | ОТСУТСТВУЕТ |")
else:
    таблица = pd.read_csv("ReviewNotes.csv").fillna("")
    отсутствующие_поля = [п for п in ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ if п not in таблица.columns]
    if отсутствующие_поля:
        ошибки.append("в ReviewNotes.csv нет колонок: " + ", ".join(отсутствующие_поля))
        строки_отчёта.append("| Таблица замечаний | нет колонок: %s |" % ", ".join(отсутствующие_поля))
    else:
        маска_заполненных = таблица[ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ].astype(str).apply(
            lambda строка: all(значение.strip() for значение in строка), axis=1)
        заполненные = таблица[маска_заполненных]
        весомых = заполненные["Серьёзность"].astype(str).str.strip().str.lower().isin(ВЕСОМЫЕ).sum()
        коротких = (заполненные["Предложение"].astype(str).str.len() < 20).sum()
        строки_отчёта.append("| Таблица замечаний | записей %d, заполненных %d, весомых %d |"
                             % (len(таблица), len(заполненные), весомых))
        if len(заполненные) < 8:
            ошибки.append("полностью заполненных замечаний %d, требуется не менее 8" % len(заполненные))
        if весомых < 3:
            ошибки.append("замечаний blocker/major %d, требуется не менее 3" % весомых)
        if коротких:
            ошибки.append("замечаний без содержательного предложения: %d" % коротких)

# --- 2. исправленный код ---
def запустить_java(имя_файла):
    if not os.path.exists(имя_файла):
        return False, "файл отсутствует"
    завершение = subprocess.run(["java", имя_файла], capture_output=True, text=True, timeout=180)
    вывод = завершение.stdout + завершение.stderr
    значимые = [с for с in вывод.splitlines() if с.strip() and not с.startswith("Picked up")]
    if завершение.returncode != 0:
        return False, "аварийное завершение: " + (значимые[0][:90] if значимые else "без сообщения")
    if "[FAIL]" in вывод:
        return False, "не пройдено проверок: %d" % вывод.count("[FAIL]")
    успешных = вывод.count("[OK]")
    if успешных < 5:
        return False, "самопроверок в файле только %d, требуется не менее 5" % успешных
    return True, "самопроверок пройдено: %d" % успешных

успех_кода, пояснение_кода = запустить_java("ShiftPlanFixed.java")
строки_отчёта.append("| `ShiftPlanFixed.java` | %s — %s |"
                     % ("пройдено" if успех_кода else "НЕ ПРОЙДЕНО", пояснение_кода))
if not успех_кода:
    ошибки.append("ShiftPlanFixed.java: " + пояснение_кода)

исходный_текст = open("ShiftPlanFixed.java", encoding="utf-8").read() if os.path.exists("ShiftPlanFixed.java") else ""
if исходный_текст and "==" in исходный_текст and re.search(r'==\s*"', исходный_текст):
    ошибки.append("в ShiftPlanFixed.java осталось сравнение строки через ==")

# --- 3. ревью на GitHub ---
ссылка_ревью = globals().get("ССЫЛКА_НА_МОЁ_РЕВЬЮ", "")
ссылка_запроса = globals().get("ССЫЛКА_НА_МОЙ_PR", "")
оставлено = int(globals().get("ЗАМЕЧАНИЙ_ОСТАВЛЕНО", 0) or 0)
получено = int(globals().get("ЗАМЕЧАНИЙ_ПОЛУЧЕНО", 0) or 0)
принято = int(globals().get("ЗАМЕЧАНИЙ_ПРИНЯТО", 0) or 0)

строки_отчёта += [
    "| Ревью на чужой запрос | %s |" % (ссылка_ревью or "НЕ УКАЗАНО"),
    "| Свой запрос | %s |" % (ссылка_запроса or "НЕ УКАЗАН"),
    "| Замечаний оставлено / получено / принято | %d / %d / %d |" % (оставлено, получено, принято),
    "| Таблица прослеживаемости | %s |" % ("заполнена" if ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА else "НЕ ЗАПОЛНЕНА"),
    "| Вопросы и выводы | %s |" % ("заполнены" if ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ else "НЕ ЗАПОЛНЕНЫ"),
]

for подпись, ссылка in [("ревью", ссылка_ревью), ("своего запроса", ссылка_запроса)]:
    if not ОБРАЗЕЦ_ССЫЛКИ.search(str(ссылка)):
        ошибки.append("ссылка %s не похожа на адрес pull request" % подпись)
if оставлено < 5:
    ошибки.append("оставлено замечаний %d, требуется не менее 5" % оставлено)
if получено and принято > получено:
    ошибки.append("принято замечаний больше, чем получено")
if not ФИО_исполнителя.strip() or not Группа_исполнителя.strip():
    ошибки.append("не указаны ФИО и группа")
if not ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА:
    ошибки.append("не заполнена таблица прослеживаемости")
if not ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ:
    ошибки.append("не заполнены контрольные вопросы и выводы")

print("\n".join(строки_отчёта))
print("\nСкопируйте отчёт выше в описание pull request.\n")

assert not ошибки, "Работа не готова к сдаче: " + "; ".join(ошибки)
print("Самопроверка пройдена: работа готова к сдаче.")

### Отчёт самопроверки «Code review»

| Параметр | Значение |
|---|---|
| Исполнитель | Дёшева Виолетта Евгеньевна, группа ГИ-41 |
| Дата отчёта (UTC) | 2026-11-27 09:37 |
| Таблица замечаний | записей 9, заполненных 9, весомых 7 |
| `ShiftPlanFixed.java` | пройдено — самопроверок пройдено: 8 |
| Ревью на чужой запрос | [https://github.com/VeronikaZmushko/Lab8_Code_Review/pull/1] |
| Свой запрос | https://github.com/Violaveta/docker_lab/pull/4 |
| Замечаний оставлено / получено / принято | 5 / 5 / 5 |
| Таблица прослеживаемости | заполнена |
| Вопросы и выводы | заполнены |

Скопируйте отчёт выше в описание pull request.

Самопроверка пройдена: работа готова к сдаче.
